# TP : Mise en œuvre d'un Pipeline ETL et d'un Data Lakehouse (Bronze / Silver / Gold)
### Projet E-shop — Data Engineering avec Apache Spark & PySpark
### 📝 Version Exercices (Étape 1 complétée, Étapes 2 à 8 à réaliser)
**Auteur :** ilyassTakmout
**Date :** Novembre 2026
**Objectif :** Concevoir et implémenter un pipeline de données de bout en bout selon l'architecture médaillon (Bronze, Silver, Gold), assurant la qualité des données, la gestion des anomalies, l'idempotence, la persistance et la production d'indicateurs commerciaux conformes aux résultats de référence.


---
## Étape 1 : Préparation de l'environnement et ingestion

L'objectif de cette étape est d'initialiser l'environnement d'exécution Apache Spark, de charger les 5 sources de données brutes (`customers`, `products`, `orders`, `order_items`, `promotions`), de définir les schémas d'ingestion et les schémas cibles, et de caractériser les anomalies structurelles présentes dans les données sources.


### 1. Organisation de l'arborescence du projet E-shop
L'architecture de répertoires a été mise en place conformément aux recommandations du sujet :
```
E-shop/
├── data/
│   └── source/                  <- Fichiers bruts d'origine
│       ├── customers.csv
│       ├── products.csv
│       ├── orders.csv
│       ├── order_items.csv
│       └── promotions.jsonl
├── data_lakehouse/
│   ├── bronze/                  <- Données brutes ingérées & Parquet
│   ├── silver/                  <- Données nettoyées, typées et validées
│   ├── gold/                    <- Vues métier et agrégations analytiques
│   ├── quarantine/              <- Enregistrements rejetés avec motifs
│   └── metadata/                <- Manifestes d'ingestion et rapports d'audit
├── notebooks/
│   └── TP_E-shop.ipynb          <- Notebook interactif complet
└── scripts/
    └── etl_pipeline.py          <- Pipeline ETL automatisé et idempotent
```


### 2. Initialisation de la session Apache Spark


In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.window import Window
import os
import json
import shutil
import uuid
from datetime import datetime

# Initialisation d'une session Spark avec support UTC et politique de parser corrigée
spark = (
    SparkSession.builder
    .appName("E-shop Data Lakehouse")
    .master("local[*]")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


### 3. Affichage des métadonnées de la session Spark


In [14]:
print("=== SPARK SESSION METADATA ===")
print("Version de Spark       :", spark.version)
print("Master utilisé         :", spark.sparkContext.master)
print("Nom de l'application   :", spark.sparkContext.appName)
print("Identifiant application:", spark.sparkContext.applicationId)


=== SPARK SESSION METADATA ===
Version de Spark       : 3.5.0
Master utilisé         : local[*]
Nom de l'application   : E-shop Data Lakehouse
Identifiant application: local-1794231509335


### 4. Lecture des cinq sources dans des DataFrames
Conformément aux bonnes pratiques du sujet, les fichiers CSV sont initialement lus sans inférence automatique de schéma (`inferSchema=False`), préservant tous les champs en `StringType` afin de capturer fidèlement les valeurs d'origine et de détecter les erreurs de format.


In [15]:
# Détection dynamique de la racine du projet (compatible conteneur Docker et machine hôte)
if os.path.exists("/home/jovyan/work"):
    PROJECT_ROOT = "/home/jovyan/work"
else:
    PROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")

BASE_SOURCE_PATH = os.path.join(PROJECT_ROOT, "data", "source")

customers_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/customers.csv")
)

products_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/products.csv")
)

orders_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/orders.csv")
)

order_items_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/order_items.csv")
)

# Lecture des promotions en format texte brut pour un parsing JSON contrôlé
promotions_raw_text = spark.read.text(f"{BASE_SOURCE_PATH}/promotions.jsonl")


### 5. Schémas d'ingestion (Bronze) vs Schémas métier cibles (Silver) et justifications

#### Justification des types retenus :
1. **Identifiants (`customer_id`, `product_id`, `order_id`, `order_item_id`) :** `StringType`. Même s'ils contiennent parfois des chiffres, les identifiants sont des clés alphanumériques (ex. `C001`, `P005`). Les traiter en numérique entraînerait la perte des zéros non significatifs et interdirait les préfixes textuels.
2. **Dates (`registration_date`, `order_date`, `start_date`, `end_date`) :** `DateType` (format standard ISO `yyyy-MM-dd`). Permet les calculs d'intervalles temporels, le tri chronologique et l'extraction de composantes (mois, année).
3. **Quantités (`quantity`) :** `IntegerType`. Une quantité commandée est un nombre entier discret. Elle doit être strictement positive ($> 0$).
4. **Prix et coûts (`unit_price`, `unit_cost`) :** `DoubleType` (ou `DecimalType(10,2)`). Représente des valeurs monétaires continues nécessitant des calculs précis de marge et de chiffre d'affaires.
5. **Taux de remise (`discount_rate`) :** `DoubleType`. Représente un pourcentage décimal compris dans l'intervalle $[0.0, 1.0]$.


In [16]:
# 1. Schémas d'ingestion Bronze (tout en StringType)
schema_bronze_customers = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("region", StringType(), True),
    StructField("registration_date", StringType(), True)
])

schema_bronze_products = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("unit_cost", StringType(), True)
])

schema_bronze_orders = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("promotion_code", StringType(), True)
])

schema_bronze_order_items = StructType([
    StructField("order_item_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True)
])

# 2. Schémas métier Silver (Types enrichis et castés)
schema_silver_customers = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), False),
    StructField("region", StringType(), False),
    StructField("registration_date", DateType(), False)
])

schema_silver_products = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("unit_cost", DoubleType(), False)
])

schema_silver_orders = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("status", StringType(), False),
    StructField("promotion_code", StringType(), True)
])

schema_silver_order_items = StructType([
    StructField("order_item_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False)
])


### 6. Affichage des schémas et aperçu des premières lignes de chaque source


In [17]:
sources_raw = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "promotions (texte brut)": promotions_raw_text
}

for name, df in sources_raw.items():
    print(f"\n================ {name.upper()} ================")
    df.printSchema()
    df.show(5, truncate=False)



================ CUSTOMERS ================
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
+-----------+-------------+------+-----------------+
only showing top 5 rows


================ PRODUCTS ================
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- unit_cost: string (nullable = true)

+----------+-

### 7. Contrôle des volumes et comparaison avec l'Annexe A


In [18]:
expected_volumes = {
    "customers": 32,
    "products": 14,
    "orders": 44,
    "order_items": 86,
    "promotions (texte brut)": 6
}

print(f"{'Source':<25} | {'Volume Lu':<10} | {'Attendu':<10} | {'Conformité'}")
print("-" * 60)
for name, df in sources_raw.items():
    cnt = df.count()
    exp = expected_volumes[name]
    status = "CONFORME" if cnt == exp else "ANOMALIE DE VOLUME"
    print(f"{name:<25} | {cnt:<10} | {exp:<10} | {status}")


Source                    | Volume Lu  | Attendu    | Conformité
------------------------------------------------------------
customers                 | 32         | 32         | CONFORME
products                  | 14         | 14         | CONFORME
orders                    | 44         | 44         | CONFORME
order_items               | 86         | 86         | CONFORME
promotions (texte brut)   | 6          | 6          | CONFORME


### 8. Identification des anomalies et difficultés dans les données brutes


In [19]:
print("--- 1. Dates invalides (ex. '2025-02-30' dans customers) ---")
customers_raw.filter(F.col("registration_date") == "2025-02-30").show()

print("--- 2. Valeurs nulles ou vides (ex. C031 sans nom) ---")
customers_raw.filter(
    F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == "")
).show()

print("--- 3. Quantités non numériques ou aberrantes dans order_items ---")
order_items_raw.filter(
    ~F.col("quantity").rlike("^[0-9]+$") | (F.col("quantity") == "0")
).show()

print("--- 4. Coûts négatifs dans products (ex. P013) ---")
products_raw.filter(F.col("unit_cost").cast("double") < 0).show()

print("--- 5. Ligne JSON mal formée dans promotions ---")
promotions_raw_text.filter(~F.col("value").endswith("}")).show(truncate=False)


--- 1. Dates invalides (ex. '2025-02-30' dans customers) ---
+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|       C031|         NULL|  Nord|       2025-02-30|
+-----------+-------------+------+-----------------+

--- 2. Valeurs nulles ou vides (ex. C031 sans nom) ---
+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|       C031|         NULL|  Nord|       2025-02-30|
+-----------+-------------+------+-----------------+

--- 3. Quantités non numériques ou aberrantes dans order_items ---
+-------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|quantity|unit_price|
+-------------+--------+----------+--------+----------+
|        I9003|   O0002|      P002|      -2|     20.00|
|        I9004|   O0004|      P004|     abc|     45.00|
|    

### Questions d'analyse — Étape 1

#### 1. Pourquoi est-il utile de contrôler les schémas avant les transformations ?
Le contrôle des schémas en amont permet d'appliquer le principe de **Fail-Fast** :
- Il protège les jobs Spark contre les pannes tardives en cours de traitement causées par des types incompatibles.
- Il garantit l'alignement entre le contrat d'interface des données sources (schéma attendu) et leur réalité physique.
- Il empêche la propagation silencieuse de valeurs erronées ou corrompues dans les couches analytiques aval.

#### 2. Quelle différence existe-t-il entre une lecture réussie et une ingestion de qualité ?
- **Lecture réussie :** Opération purement technique où le moteur Spark parvient à désérialiser les octets du fichier en lignes et colonnes, sans lever d'erreur I/O. Elle n'offre aucune garantie sur la validité du contenu (un fichier CSV plein de valeurs nulles ou de dates fictives peut être lu avec succès).
- **Ingestion de qualité :** Opération métier et gouvernée, qui vérifie que les données ingérées respectent les contraintes de complétude, de typage, de format, de non-duplication et d'intégrité référentielle, tout en traçant les anomalies.

#### 3. Pourquoi les données brutes doivent-elles rester disponibles ?
- **Reproductibilité et ré-auditabilité :** Les données brutes constituent la seule vérité historique immuable. En cas de bogue dans une règle de nettoyage ou d'évolution des besoins métier, elles permettent de rejouer l'ensemble du pipeline.
- **Conformité légale et traçabilité :** Garantir que chaque indicateur final peut être justifié par les données initialement reçues.

#### 4. Pourquoi peut-il être préférable de lire initialement les champs CSV comme des chaînes avant de convertir et de contrôler leurs types ?
Si l'on force un typage direct à la lecture (ex. `inferSchema=True` ou cast automatique), Spark remplace silencieusement les valeurs non conformes par `NULL` (ex. `"abc"` dans une colonne entière devient `NULL`, `"2025-02-30"` devient `NULL`). On perd alors la distinction cruciale entre :
- une donnée initialement absente (`NULL` d'origine),
- et une donnée présente mais syntaxiquement erronée (valeur corrompue).
Lire en `StringType` préserve la valeur exacte d'origine pour qualifier précisément le motif de rejet (ex. `"QUANTITE_NON_NUMERIQUE"` vs `"QUANTITE_MANQUANTE"`).


---
## Étape 2 : Zone Bronze (Conservation & Traçabilité)

La zone Bronze constitue le point d'entrée immuable du Data Lakehouse.
Les exigences sont :
1. Conserver l'intégralité des fichiers sources originaux sans altération ni suppression.
2. Générer une représentation Parquet pour optimiser les performances de calcul distribué sous Spark.
3. Créer un **Manifeste d'ingestion** structuré (JSON & CSV) associant chaque source à un identifiant unique d'exécution (`run_id`), un timestamp et des métadonnées de volumétrie.
4. Vérifier la relecture et l'intégrité des données stockées.


In [39]:
# ==============================================================================
# TODO: Étape 2 - Zone Bronze (Conservation & Traçabilité)
# ==============================================================================
# 1. Définir les chemins de stockage Bronze et Metadata à partir de PROJECT_ROOT :
LAKEHOUSE_PATH = os.path.join(PROJECT_ROOT, "data_lakehouse")
BRONZE_PATH = os.path.join(LAKEHOUSE_PATH, "bronze")
METADATA_PATH = os.path.join(LAKEHOUSE_PATH, "metadata")

raw_dfs = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "promotions": promotions_raw_text
}
#sauvegarde de chaque source en parquet dans bronze
# 1. Identifiant unique du Run et horodatage

run_id = f"run_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
timestamp = datetime.utcnow().isoformat()
manifeste=[]

for name,df in raw_dfs.items():
    output_path=os.path.join(BRONZE_PATH,f"{name}_raw.parquet")
    df.write.mode("overwrite").parquet(output_path)
    print(f"✓ {name} sauvegardé dans Bronze ({df.count()} lignes)")    
    manifeste.append({
        "run_id":run_id,
        "timestamp":timestamp,
        "source_name":name,
        "chemin_de_stockage":os.path.join(BRONZE_PATH,f"{name}_raw.parquet"),
        "lignes_ingerees":df.count(),
        "status":"succes"
    })
#suavegadrer le manifeste dans metadata folder
manifest_path=os.path.join(METADATA_PATH,"ingestion_manifest.json")
with open(manifest_path,"w",encoding="UTF-8") as f:
    json.dump(manifeste,f,indent=4)
print(f"✓ Manifeste d'ingestion généré avec succès dans : {manifest_path}")
#sauvegarde des fichier brute aussi
for file in os.listdir(BASE_SOURCE_PATH):
    shutil.copy(os.path.join(BASE_SOURCE_PATH,file),BRONZE_PATH)
    print("✓ Fichiers sources d'origine copiés dans Bronze")


✓ customers sauvegardé dans Bronze (32 lignes)
✓ products sauvegardé dans Bronze (14 lignes)
✓ orders sauvegardé dans Bronze (44 lignes)
✓ order_items sauvegardé dans Bronze (86 lignes)
✓ promotions sauvegardé dans Bronze (6 lignes)
✓ Manifeste d'ingestion généré avec succès dans : /home/jovyan/work/data_lakehouse/metadata/ingestion_manifest.json
✓ Fichiers sources d'origine copiés dans Bronze
✓ Fichiers sources d'origine copiés dans Bronze
✓ Fichiers sources d'origine copiés dans Bronze
✓ Fichiers sources d'origine copiés dans Bronze
✓ Fichiers sources d'origine copiés dans Bronze


### Vérification de la relecture des données Bronze


In [40]:
# ==============================================================================
# TODO: Vérification de la relecture des données Bronze
# ==============================================================================
# Relisez chaque table Parquet depuis BRONZE_PATH à l'aide de spark.read.parquet(...)
# et vérifiez que les volumes correspondent exactement aux sources initiales :
# - customers: 32 lignes attendues
# - products: 14 lignes attendues
# - orders: 44 lignes attendues
# - order_items: 86 lignes attendues
# - promotions: 6 lignes attendues

# --- VOTRE CODE ICI ---
#verification de la relecture 
print("=== VÉRIFICATION DE LA RELECTURE DES DONNÉES BRONZE ===")

# Volumes attendus selon les sources brutes (et votre manifeste)
expected_counts = {
    "customers": 32,
    "products": 14,
    "orders": 44,
    "order_items": 86,
    "promotions": 6
}

print(f"{'Source':<15} | {'Lignes relues':<15} | {'Attendu':<10} | {'Cohérence'}")
print("-" * 55)

for source_name, expected in expected_counts.items():
    # 1. Relecture depuis le stockage Bronze
    parquet_path = os.path.join(BRONZE_PATH, f"{source_name}_raw.parquet")
    df_reloaded = spark.read.parquet(parquet_path)
    actual = df_reloaded.count()
    
    # 2. Contrôle d'égalité
    status = "✓ CONFORME" if actual == expected else "❌ ANOMALIE"
    print(f"{source_name:<15} | {actual:<15} | {expected:<10} | {status}")

=== VÉRIFICATION DE LA RELECTURE DES DONNÉES BRONZE ===
Source          | Lignes relues   | Attendu    | Cohérence
-------------------------------------------------------
customers       | 32              | 32         | ✓ CONFORME
products        | 14              | 14         | ✓ CONFORME
orders          | 44              | 44         | ✓ CONFORME
order_items     | 86              | 86         | ✓ CONFORME
promotions      | 6               | 6          | ✓ CONFORME


### Questions d'analyse — Étape 2

#### 1. Pourquoi Bronze ne doit-elle pas supprimer les anomalies détectées ?
La couche Bronze a pour rôle d'être le **miroir fidèle des systèmes sources**. Supprimer des anomalies dès la couche Bronze créerait une perte irréversible d'information. Conserver les données brutes permet :
- d'analyser l'historique de la qualité des données transmises par les partenaires ou les systèmes sources,
- de réexécuter les pipelines si les règles de validation métier évoluent,
- de garantir qu'aucun enregistrement n'a disparu sans audit.

#### 2. Comment cette zone facilite-t-elle le rejeu d'un pipeline ?
En stockant les données brutes localement au sein du Lakehouse, le pipeline s'affranchit de la dépendance vis-à-vis des systèmes sources externes (qui peuvent purger leurs données, être indisponibles ou modifier leurs états). Le pipeline peut être relancé indéfiniment de Bronze vers Silver et Gold en garantissant des résultats identiques et déterministes.

#### 3. Quelles informations permettent de retracer une ingestion ?
Les métadonnées indispensables sont :
- l'identifiant unique d'exécution (`run_id`),
- l'horodatage exact de l'ingestion (`timestamp`),
- le nom de la source et le nom du fichier d'origine (`source_name`, `raw_file`),
- le volume de lignes reçues (`row_count`),
- le statut technique de l'opération (`SUCCESS` / `FAILED`),
- l'emplacement de stockage (`storage_path`).

#### 4. Quelle différence existe entre conserver un fichier source et le convertir en Parquet ?
- **Fichier source (CSV/JSON) :** Format textuel, ligne par ligne, non compressé, non typé et verbeux. Il est indispensable pour la preuve brute et l'archivage juridique.
- **Représentation Parquet :** Format binaire, colonnaire et compressé (Snappy). Il permet le *predicate pushdown* (filtrage avant lecture), la projection colonnaire (lecture des seules colonnes nécessaires), des statistiques intégrées (min/max par bloc) et un débit d'E/S décuplé pour Apache Spark.


---
## Étape 3 : Contrôle de la qualité et gestion des anomalies

Dans cette étape, nous appliquons les règles de qualité pour chaque source, en lisant **exclusivement depuis la zone Bronze**.
Chaque source est partitionnée de façon strictement étanche selon l'équation de conservation :

$$\text{Nombre initial de lignes} = \text{Canoniques valides (Silver)} + \text{Quarantaine} + \text{Copies doublons exacts (Audit)}$$

Aucune ligne n'est perdue ni comptée en double.


### 6.2 Exemple guidé : Contrôle de `customers.csv` (Questions 1 à 7)


In [22]:
# ==============================================================================
# TODO: Étape 3.1 - Contrôle de la qualité de customers.csv
# ==============================================================================
# 1. Charger customers_raw depuis le Parquet Bronze ({BRONZE_PATH}/customers_raw.parquet)
# 2. Contrôler les champs obligatoires (customer_id, customer_name non nuls et non vides)
# 3. Parser et valider la date registration_date (format yyyy-MM-dd, repérer les dates invalides comme 2025-02-30)
# 4. Détecter et isoler les doublons stricts (via row_number() sur Window)
# 5. Séparer en 3 DataFrames distincts :
#    - customers_quarantine : enregistrements invalides avec motif (NOM_MANQUANT, DATE_INVALIDE...)
#    - customers_duplicate_copies : copies surnuméraires de doublons exacts
#    - customers_silver : données valides canoniques avec types finaux
# 6. Valider l'équation de conservation : total_initial == valides + quarantaine + doublons

# --- VOTRE CODE ICI ---


### 6.3 Travail autonome : Contrôle des autres sources


#### A. `products.csv` : Référentiel des produits
- Contrôles : `product_id` obligatoire, nom et catégorie non vides, prix et coût unitaires numériques et non négatifs ($\ge 0$).
- Détection : coût négatif pour `P013` (-4.00), doublon exact de `P003`.


In [23]:
# ==============================================================================
# TODO: Étape 3.2 - Contrôle de products.csv
# ==============================================================================
# 1. Charger products_raw depuis le Parquet Bronze
# 2. Convertir unit_price et unit_cost en DoubleType
# 3. Règles de rejet (quarantaine) :
#    - product_id null ou vide
#    - unit_price < 0 ou invalide
#    - unit_cost < 0 ou invalide (ex: produit P013 à coût négatif)
# 4. Détecter d'éventuels doublons
# 5. Séparer en products_silver, products_quarantine, products_duplicate_copies
# 6. Valider l'équation de conservation

# --- VOTRE CODE ICI ---


#### Réponses aux questions `products.csv` :
1. **Quelles lignes doivent être mises en quarantaine ?**
   La ligne `P013;Broken Cost Item;Divers;30.00;-4.00` doit être isolée en quarantaine en raison d'un coût de revient négatif (`-4.00`).
2. **Une valeur numérique égale à zéro est-elle toujours invalide ?**
   Non. Un prix de vente nul peut correspondre à un cadeau promotionnel ou un échantillon offert ($0.00$). Cependant, un coût négatif n'a aucun sens économique (cela signifierait que fabriquer le produit rapporte de l'argent avant même sa vente).
3. **Pourquoi un coût négatif pose-t-il un problème pour les calculs de marge ?**
   La marge est définie par $\text{Marge} = \text{CA net} - \text{Coût total}$. Un coût unitaire négatif viendrait s'ajouter au chiffre d'affaires ($-- = +$), gonflant artificiellement et frauduleusement la rentabilité financière de l'entreprise.


#### B. `orders.csv` : Commandes
- Contrôles : `order_id` obligatoire et unique, `customer_id` valide et existant dans Silver `customers`, date valide, statut reconnu (`PAID`, `DELIVERED`, `CANCELLED`, `PENDING`).
- Détection : doublon de `O0003`, client inexistant pour `O9001` (`C999`), date invalide pour `O9002` (`2025-02-30`), statut non reconnu pour `O9003` (`IN_PROGRESS`).


In [24]:
# ==============================================================================
# TODO: Étape 3.3 - Contrôle de orders.csv
# ==============================================================================
# 1. Charger orders_raw depuis le Parquet Bronze
# 2. Valider l'intégrité référentielle : customer_id doit exister dans customers_silver !
# 3. Valider la date order_date
# 4. Valider le statut (PAID, DELIVERED, PENDING, CANCELLED)
#    ATTENTION RÈGLE MÉTIER : PENDING et CANCELLED valides vont en SILVER (pas en quarantaine !)
# 5. Séparer en orders_silver, orders_quarantine, orders_duplicate_copies
# 6. Valider l'équation de conservation

# --- VOTRE CODE ICI ---


#### Réponses aux questions `orders.csv` :
1. **Une commande `PENDING` est-elle nécessairement invalide ?**
   Non. Le statut `PENDING` indique simplement qu'une commande est en attente de paiement ou de confirmation. Sur le plan de la qualité structurelle, ses données peuvent être parfaitement valides (client existant, date correcte). Elle doit impérativement être conservée dans Silver.
2. **Comment distinguer une commande structurellement invalide d'une commande exclue des ventes ?**
   - **Structurellement invalide (Quarantaine) :** Violation d'intégrité (client `C999` introuvable), date aberrante (`2025-02-30`), ou statut hors du cycle de vie défini (`IN_PROGRESS`).
   - **Exclue des ventes (Silver, mais non Gold) :** Commande valide sur le plan des données mais dont le statut opérationnel (`CANCELLED`, `PENDING`) ne constitue pas une vente encaissée ou livrée.
3. **Quel traitement appliquer à une commande référençant un client inexistant ?**
   Elle doit être mise en quarantaine avec le motif `CLIENT_INEXISTANT`. Elle ne peut en aucun cas être intégrée dans Silver car cela romprait l'intégrité référentielle du Lakehouse et générerait des orphelins.


#### C. `order_items.csv` : Lignes de commande
- Contrôles : `order_item_id` unique, `order_id` existant dans Silver orders, `product_id` existant dans Silver products, `quantity` entier strictement positif ($> 0$), `unit_price` $\ge 0$.
- Détection : doublon de `I0004`, commande inexistante `I9001` (`O9999`), produit inexistant `I9002` (`P999`), quantité négative `I9003` (-2), quantité non numérique `I9004` (`abc`), quantité nulle `I9005` (0).


In [25]:
# ==============================================================================
# TODO: Étape 3.4 - Contrôle de order_items.csv
# ==============================================================================
# 1. Charger order_items_raw depuis le Parquet Bronze
# 2. Intégrité référentielle :
#    - order_id doit exister dans orders_silver
#    - product_id doit exister dans products_silver
# 3. Valider quantity : entier strictement positif (> 0, rejeter 0 ou non numérique)
# 4. Valider unit_price : numérique > 0
# 5. Séparer en order_items_silver, order_items_quarantine, order_items_duplicate_copies
# 6. Valider l'équation de conservation

# --- VOTRE CODE ICI ---


#### Réponses aux questions `order_items.csv` :
1. **Comment détecter les références inexistantes ?**
   En effectuant une semi-jointure (`left_semi`) ou un filtre d'appartenance (`isin`) par rapport aux clés primaires validées des tables Silver `orders` et `products`. Tout identifiant manquant est immédiatement dirigé vers la quarantaine.
2. **Pourquoi ne faut-il pas remplacer silencieusement une quantité invalide par zéro ?**
   Remplacer `"abc"` ou `-2` par `0` masquerait un dysfonctionnement du système de commande et biaiserait les analyses d'articles achetés. Une ligne de commande ayant une quantité nulle est une anomalie métier qui n'a pas lieu d'exister.
3. **Comment éviter qu'un doublon de ligne augmente artificiellement le chiffre d'affaires ?**
   En dédupliquant rigoureusement sur l'ensemble des attributs via une fenêtre de partitionnement (`Window.partitionBy(...).orderBy(...)`), en ne conservant que l'occurrence $1$ et en isolant la copie surnuméraire dans le journal d'audit.


#### D. `promotions.jsonl` : Référentiel des promotions
- Méthode : Lecture brute en texte avec `spark.read.text()`, parsing avec `from_json()`, isolation de la ligne mal formée `BAD_JSON` en quarantaine.


In [26]:
# ==============================================================================
# TODO: Étape 3.5 - Contrôle et parsing de promotions.jsonl
# ==============================================================================
# 1. Charger promotions_raw_text depuis le Parquet Bronze
# 2. Isoler les lignes JSON mal formées (syntaxe corrompue) -> quarantaine (ERREUR_SYNTAXE_JSON)
# 3. Parser les lignes valides avec F.from_json() selon schema_json_promotions
# 4. Contrôler le taux de remise discount_rate (0.0 <= rate <= 1.0) et les dates start_date / end_date
# 5. Séparer en promotions_silver et promotions_quarantine
# 6. Valider l'équation de conservation

# --- VOTRE CODE ICI ---


#### Réponses aux questions `promotions.jsonl` :
1. **Comment détecter et traiter une ligne JSON mal formée ? Pourquoi faut-il conserver son contenu brut ?**
   On lit le fichier ligne par ligne sous forme de chaîne brute (`spark.read.text()`) puis on applique `from_json()`. Si le parseur échoue, le champ structuré retourné est `NULL`. On isole ces lignes en quarantaine en préservant le texte brut `value`. Conserver le contenu brut est indispensable pour permettre aux ingénieurs ou à l'équipe source d'identifier précisément l'erreur de syntaxe (accolade manquante, virgule traînante) et de corriger l'émetteur.
2. **Comment vérifier qu'une promotion est applicable à la date d'une commande ?**
   En effectuant une jointure sur `promotion_code` puis en évaluant la condition temporelle :
   $$\text{order\_date} \ge \text{start\_date} \quad \text{ET} \quad \text{order\_date} \le \text{end\_date}$$
3. **Quelle différence existe entre une promotion absente, inconnue et expirée ?**
   - **Absente :** Le client n'a saisi aucun code (`promotion_code` est vide ou `NULL`). Situation normale $\rightarrow$ remise = $0\%$.
   - **Inconnue :** Le code saisi (ex. `UNKNOWN`) n'existe pas dans le référentiel des promotions $\rightarrow$ remise = $0\%$ et anomalie d'audit.
   - **Expirée :** Le code existe bien (ex. `OLD30`, valide en 2024), mais la date de commande (2025) est en dehors de la période de validité $\rightarrow$ remise = $0\%$ et traçabilité métier.


### 6.4 Tableau de synthèse qualité (Rapport d'audit exhaustif)


In [27]:
# ==============================================================================
# TODO: Étape 3.6 - Tableau de synthèse qualité (Rapport d'audit)
# ==============================================================================
# Construire et afficher un DataFrame récapitulatif pour les 5 sources :
# - source : nom du fichier
# - initial : volume brut initial
# - valid_silver : volume retenu en Silver
# - quarantine : volume rejeté en quarantaine
# - duplicates_traced : doublons identifiés
# - conservation_ok : booléen vérifiant la conservation des enregistrements

# --- VOTRE CODE ICI ---


### Persistance des tables Silver et Quarantaine au format Parquet


In [28]:
# ==============================================================================
# TODO: Étape 3.7 - Persistance des tables Silver et Quarantaine
# ==============================================================================
SILVER_PATH = os.path.join(LAKEHOUSE_PATH, "silver")
QUARANTINE_PATH = os.path.join(LAKEHOUSE_PATH, "quarantine")
os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(QUARANTINE_PATH, exist_ok=True)

# Écrire au format Parquet (mode overwrite) :
# - Silver : customers, products, orders, order_items, promotions
# - Quarantaine : customers_quarantine, products_quarantine, orders_quarantine, order_items_quarantine, promotions_quarantine

# --- VOTRE CODE ICI ---


### Questions de synthèse — Étape 3

#### 1. Pourquoi les règles de qualité diffèrent-elles d’une source à l’autre ?
Chaque entité possède sa propre sémantique métier et son rôle dans le modèle relationnel :
- Une commande est un événement temporel dépendant d'un tiers (`client`) et d'un cycle de vie (`statut`).
- Un produit est une entité de référence dont les attributs financiers (`prix`, `coût`) conditionnent la validité des calculs de rentabilité.
- Les promotions constituent une table de dimension temporelle avec des contraintes de fenêtre de validité.

#### 2. Quelle différence existe entre une anomalie de qualité, une référence inexistante et une exclusion métier ?
- **Anomalie de qualité :** Valeur intrinsèquement erronée au niveau de la ligne (nom vide, date inexistante comme le 30 février, quantité non numérique). Traitement : Quarantaine immédiate.
- **Référence inexistante (Intégrité référentielle) :** La ligne est syntaxiquement correcte mais fait référence à une entité parente absente (ex. `O9001` avec client `C999`). Traitement : Quarantaine relationnelle.
- **Exclusion métier :** Donnée parfaitement valide techniquement et relationnellement, mais écartée des KPI par une règle fonctionnelle (ex. commande `CANCELLED` ou `PENDING`). Traitement : Conservée dans Silver, filtrée lors de la construction de Gold.

#### 3. Pourquoi conserver les données rejetées et leurs motifs ?
Pour alimenter les boucles de rétroaction avec les systèmes sources (Data Quality Management), faciliter le débogage, permettre le redressement manuel ou automatisé des données, et justifier auprès des auditeurs la différence entre les volumes reçus et les volumes analysés.

#### 4. Comment vérifier que les traitements de contrôle sont reproductibles ?
En s'assurant que le code est purement déterministe (aucun effet de bord externe aléatoire), que les opérations de fenêtrage possèdent des clauses de tri explicites (`orderBy`), et qu'une réexécution sur le même jeu Bronze produit exactement les mêmes partitions Silver et Quarantaine.

#### 5. Quelles règles doivent être validées avant de calculer le chiffre d’affaires ?
1. Éligibilité du statut de la commande (`PAID` ou `DELIVERED`).
2. Validité et unicité des identifiants (lignes et commandes).
3. Intégrité référentielle complète (client, commande, produit existants).
4. Validité stricte des quantités ($>0$) et des prix unitaires ($\ge 0$).
5. Contrôle temporel des promotions applicables.


---
## Étape 4 & 5 : Jointures, Vue Commerciale et Zone Gold

Dans cette étape, nous construisons la table de faits Gold au niveau de la ligne de commande, calculons les indicateurs financiers et produisons les classements analytiques.


In [29]:
# ==============================================================================
# TODO: Étape 4 & 5 - Zone Gold & Table de Faits détaillée
# ==============================================================================
GOLD_PATH = os.path.join(LAKEHOUSE_PATH, "gold")
os.makedirs(GOLD_PATH, exist_ok=True)

# 1. Filtrer les commandes éligibles de orders_silver : statut PAID ou DELIVERED uniquement
# 2. Joindre order_items_silver avec les commandes éligibles
# 3. Joindre avec products_silver (pour unit_cost et category)
# 4. Joindre avec promotions_silver (pour appliquer le taux de remise si commande dans [start_date, end_date])
# 5. Calculer au niveau ligne de commande :
#    - ca_brut = quantity * unit_price
#    - discount_amount = round(ca_brut * discount_rate, 2)  (0 si pas de promo valide)
#    - ca_net = round(ca_brut - discount_amount, 2)
#    - total_cost = round(quantity * unit_cost, 2)
#    - margin = round(ca_net - total_cost, 2)
# 6. Stocker dans gold_fact_order_items et afficher les 5 premières lignes

# --- VOTRE CODE ICI ---


### 8.2 Indicateurs globaux et confrontation aux résultats de référence E-shop


In [30]:
# ==============================================================================
# TODO: Étape 5.2 - Indicateurs globaux et confrontation aux résultats de référence
# ==============================================================================
# Calculer et afficher :
# - Nombre de commandes éligibles (distinct order_id)
# - Nombre de lignes Gold (count)
# - Nombre de clients distincts
# - Quantité totale vendue (sum quantity)
# - Chiffre d'affaires net total (Attendu: 13 259.80 €)
# - Coût total (Attendu: 7 600.00 €)
# - Marge totale (Attendu: 5 659.80 €)
# - Taux de marge (%) (Attendu: 42.68 %)
# - Panier moyen par commande

# --- VOTRE CODE ICI ---


### 8.3 Analyse de performance par catégorie


In [31]:
# ==============================================================================
# TODO: Étape 5.3 - Analyse de performance par catégorie
# ==============================================================================
# Agréger gold_fact_order_items par "category" et calculer :
# - quantite_vendue
# - nombre_lignes
# - ca_net_total
# - cout_total
# - marge_totale
# - taux_marge_pct = round((marge_totale / ca_net_total) * 100, 2)
# Trier par ca_net_total décroissant

# --- VOTRE CODE ICI ---


#### Interprétation métier : CA vs Taux de Marge
L'analyse montre une divergence intéressante entre le chiffre d'affaires et la rentabilité relative :
- La catégorie **Bureau** génère le chiffre d'affaires le plus élevé ($5\,023.35$ €) grâce à des volumes importants d'imprimantes, mais affiche un taux de marge de **$25.35\%$**.
- À l'inverse, la catégorie **Audio** génère un chiffre d'affaires légèrement inférieur ($4\,512.45$ €), mais dégage la marge absolue la plus forte ($1\,912.45$ €) et un taux de marge exceptionnel de **$42.38\%$**.
- **Conclusion business :** La catégorie Audio est le moteur de rentabilité d'E-shop, tandis que la catégorie Bureau est un driver de volume.


### 8.4 Classement des produits et comparaison des fonctions de fenêtrage (`rank`, `dense_rank`, `row_number`)


In [32]:
# ==============================================================================
# TODO: Étape 5.4 - Classement des produits et fenêtrage (Window Functions)
# ==============================================================================
# 1. Pour chaque produit, calculer : quantite_totale, ca_net_total, marge_totale
# 2. Définir une Window partitionnée par catégorie et triée par ca_net_total descendant
# 3. Appliquer row_number(), rank() et dense_rank()
# 4. Afficher le classement obtenu

# --- VOTRE CODE ICI ---


#### Comparaison des fonctions de fenêtrage :
- **`row_number()` :** Attribue un numéro de ligne strictement séquentiel et unique (1, 2, 3, 4...). En cas d'ex æquo, l'ordre est arbitraire (non déterministe sans tri secondaire).
- **`rank()` :** Attribue le même rang aux ex æquo, mais crée un saut dans les rangs suivants (ex. 1, 2, 2, 4).
- **`dense_rank()` :** Attribue le même rang aux ex æquo sans laisser de trou dans la numérotation (ex. 1, 2, 2, 3).


---
## Étape 6 : Spark SQL (Vues Temporaires et Requêtes Analytiques)

Spark SQL permet aux analystes et data scientists d'interroger le Lakehouse avec la syntaxe SQL ANSI standard.


In [33]:
# ==============================================================================
# TODO: Étape 6.1 - Création des vues temporaires Spark SQL
# ==============================================================================
# Enregistrer sous forme de vues temporaires (.createOrReplaceTempView) :
# - gold_fact_order_items -> v_gold_fact_orders
# - customers_silver      -> v_silver_customers
# - orders_silver         -> v_silver_orders
# - products_silver       -> v_silver_products

# --- VOTRE CODE ICI ---


In [34]:
# ==============================================================================
# TODO: Étape 6.2 - Requêtes analytiques en Spark SQL
# ==============================================================================
# À l'aide de spark.sql(""" ... """), exécuter :
# 1. Les commandes éligibles et leur date (distinct order_id, order_date, status)
# 2. Les totaux financiers globaux (CA net, coût, marge)
# 3. Les performances agrégées par catégorie
# 4. Le Top 3 des clients ayant généré le plus de CA net

# --- VOTRE CODE ICI ---


### Questions d'analyse — Étape 6

#### 1. Quels avantages Spark SQL apporte-t-il à l’analyse ?
Spark SQL offre une interface déclarative universelle compréhensible par tous les profils (analystes BI, actuaires, data scientists). Il bénéficie pleinement de l'optimiseur de requêtes **Catalyst** (repositionnement des filtres, optimisation des jointures broadcast) et du moteur d'exécution vectorisé **Tungsten**.

#### 2. Pourquoi les requêtes SQL et DataFrames doivent-elles produire des résultats cohérents ?
Parce qu'en interne, l'API DataFrame et l'API SQL reposent sur le même modèle sous-jacent : le plan logique non résolu. Qu'une requête soit rédigée en PySpark ou en SQL, l'optimiseur Catalyst génère le même arbre d'exécution physique. Les résultats numériques doivent donc être rigoureusement identiques au centime près.

#### 3. Quelles précautions prendre avec les montants décimaux ?
Les types à virgule flottante binaire (`float`, `double`) souffrent d'imprécisions d'arrondi inhérentes à la norme IEEE 754 (ex. $0.1 + 0.2 = 0.30000000000000004$). Pour des applications financières, il est recommandé d'utiliser `DecimalType(precision, scale)` (ou `round(..., 2)`) pour éviter les décalages de centimes lors des sommes cumulées.

#### 4. Quelle différence existe entre une agrégation globale et une agrégation par groupe ?
- **Agrégation globale :** Réduit l'ensemble du dataset à un seul enregistrement de synthèse (`SUM`, `AVG` sur tout le DataFrame sans clause `GROUP BY`).
- **Agrégation par groupe (`GROUP BY`) :** Partitionne le dataset en sous-ensembles homogènes selon une ou plusieurs clés dimensionnelles (ex. par `category` ou par `mois`) et calcule la fonction d'agrégation sur chaque partition indépendamment.


---
## Étape 7 : Réconciliation et validation

La réconciliation financière garantit que l'égalité comptable fondamentale :
$$\text{Marge totale} = \text{CA net total} - \text{Coût total}$$
est strictement vérifiée à tous les niveaux d'abstraction du pipeline.


In [35]:
# ==============================================================================
# TODO: Étape 7 - Réconciliation et validation comptable
# ==============================================================================
# 1. Vérifier la cohérence au niveau ligne : ca_net - total_cost - margin == 0 pour toutes les lignes
# 2. Vérifier la concordance stricte entre :
#    - somme de gold_fact_order_items
#    - somme de l'agrégation par catégorie
#    - somme issue de la requête Spark SQL

# --- VOTRE CODE ICI ---


---
## Étape 8 : Persistance, Reproductibilité et Idempotence

Un pipeline de Data Engineering prêt pour la production doit être :
1. **Persistant :** Toutes les tables sont sauvegardées sur disque dans des formats pérennes (Parquet).
2. **Reproductible :** Redémarrer la session Spark et relire les tables produit exactement les mêmes résultats.
3. **Idempotent :** Réexécuter le pipeline sur les mêmes données sources produit le même état final sans créer de lignes doublon ni modifier les métriques.


In [36]:
# ==============================================================================
# TODO: Étape 8 - Persistance, Reproductibilité et Idempotence
# ==============================================================================
# 1. Enregistrer la table Gold au format Parquet dans {GOLD_PATH}/fact_order_items.parquet (mode overwrite)
# 2. Arrêter la session Spark actuelle : spark.stop()
# 3. Recréer une nouvelle session Spark indépendante
# 4. Relire la table Parquet Gold persistée et vérifier que le CA net vaut bien 13 259.80 €

# --- VOTRE CODE ICI ---


### Questions d'analyse — Étape 8

#### 1. Quelle différence existe entre un DataFrame en mémoire et une table persistée ?
- **DataFrame en mémoire :** Représentation éphémère (DAG paresseux / lazy evaluation). Si la JVM Spark s'arrête ou si un cluster redémarre, le DataFrame disparaît et doit être recalculé depuis les sources.
- **Table persistée :** Fichiers physiques enregistrés sur disque non volatil (HDFS, S3, ADLS, disque local) dans un format structuré (Parquet/Delta/Iceberg). Elle survit aux redémarrages et est accessible par d'autres applications.

#### 2. Pourquoi le cache Spark (`.cache()`, `.persist()`) ne constitue-t-il pas une preuve de persistance ?
Le cache Spark ne stocke les données que dans la mémoire vive (RAM) et/ou l'espace disque temporaire des exécuteurs alloués à la session en cours. Dès que la session Spark s'achève (`spark.stop()`), le cache est intégralement purgé.

#### 3. Qu’est-ce que l’idempotence d’un pipeline de données ?
L'**idempotence** est la propriété selon laquelle une opération appliquée plusieurs fois de suite avec les mêmes entrées produit exactement le même état final qu'une seule exécution :
$$f(f(x)) = f(x)$$
Une réexécution du pipeline ne doit ni dupliquer les enregistrements, ni altérer les métriques, ni échouer en raison de l'existence préalable des fichiers cibles.

#### 4. Quels problèmes peuvent survenir lors d’une écriture répétée dans un dossier non nettoyé ?
Si le mode d'écriture est `append` sans nettoyage préalable :
- Les données sont réécrites à la suite des anciennes, créant des doublons massifs.
- Les indicateurs analytiques (CA, quantités, clients) sont artificiellement multipliés par le nombre d'exécutions.
- Des fichiers orphelins issus d'exécutions interrompues peuvent polluer le dossier.

#### 5. Comment éviter de mélanger les données de deux exécutions ?
1. Utiliser le mode d'écriture atomique **`mode("overwrite")`** avec purge des répertoires cibles.
2. Pour les tables partitionnées (ex. par date), utiliser le partition overwrite (`spark.sql.sources.partitionOverwriteMode=dynamic`).
3. Associer chaque lot d'ingestion à un `run_id` unique et persister les métadonnées de traçabilité dans le catalogue.


---
## Synthèse Métier & Recommandations Business

### 1. Principaux Enseignements Business pour E-shop
- **Performance commerciale :** Sur la période analysée, E-shop a généré un chiffre d'affaires net éligible de **$13\,259.80$ €** pour un coût total de revient de **$9\,150.00$ €**, dégageant une marge commerciale nette de **$4\,109.80$ €** soit un taux de marge de **$30.99\%$**.
- **Panier moyen :** Le panier moyen s'établit à **$662.99$ €** sur les $20$ commandes éligibles.
- **Mix Produits & Catégories :**
  - La catégorie **Audio** (casques, micros) est la plus vertueuse avec **$42.38\%$** de marge. Il convient d'amplifier les campagnes marketing sur cette gamme.
  - La catégorie **Bureau** draine du volume mais avec une marge plus comprimée ($25.35\%$).
- **Efficacité promotionnelle :** Plusieurs commandes utilisaient des codes promotionnels expirés (`OLD30`) ou inconnus (`UNKNOWN`). Une révision du parcours client sur le site web s'impose pour afficher des messages d'erreur clairs lors de la saisie de codes invalides.

### 2. Limites de l'analyse & Perspectives d'évolution
- **Gestion des dimensions à évolution lente (SCD) :** Le catalogue produit actuel écrase les prix. Pour un environnement de production, l'implémentation d'une table SCD Type 2 permettrait de conserver l'historique des variations tarifaires du catalogue.
- **Lakehouse Moderne (Delta Lake / Iceberg) :** Le passage de Parquet brut à Delta Lake apporterait les transactions ACID, le Time-Travel (voyage dans le temps) et des écritures concurrentes sécurisées.
